In [1]:
# Import libraries
import numpy as np
from PIL import Image
from pathlib import Path
import umap
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import re
from enum import Enum

/Users/matthewlaujh/Desktop/desktop/itpnyu/itpnyu-thesis/itpnyu-thesis/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class ProcessingMode(Enum):
    SINGLE = "single"
    MULTIPLE = "multiple"
    COMBINED = "combined"
    
CONFIG = {
    # Paths
    'paths': {
        'project_root': Path('/Users/matthewlaujh/Desktop/desktop/itpnyu/itpnyu-thesis/itpnyu-thesis/prod/fenway'),
        'image_dir': Path('/Users/matthewlaujh/Desktop/desktop/itpnyu/itpnyu-thesis/itpnyu-thesis/prod/images/fenway'),
        'output_dir': Path('/Users/matthewlaujh/Desktop/desktop/itpnyu/itpnyu-thesis/itpnyu-thesis/prod/output/fenway'),
        'output_prefix': 'umap_data',  # Base prefix for all output files
    },
    
    # Image Processing
    'image': {
        'resize_width': 300,
        'allowed_extensions': ['.jpg', '.jpeg', '.png']
    },
    
    # UMAP Parameters
    'umap': {
        '2d': {
            'n_components': 2,
            'n_neighbors': 15,
            'min_dist': 1,
            'metric': 'euclidean',
            'random_state': 42
        },
        '3d': {
            'n_components': 3,
            'n_neighbors': 15,
            'min_dist': 0.1,
            'metric': 'euclidean',
            'random_state': 42
        }
    },
    
    # LED Matrix
    'led': {
        'matrix_size': 8,
        'brightness': 255,
    },

    'visualization': {
        'plot_umap': False,      # Plot UMAP embeddings
        'preview_matrix': False, # Preview LED matrix
        'preview_animation': False # Preview animations
    },
    
    # Output Files
    'output': {
        'matrix_suffix': '',             # For regular LED matrix files
        'animation_suffix': '_animation',  # For animation files
        'multiple_suffix': '_multiple',    # For multiple mode files
        'combined_suffix': '_combined'     # For combined mode files
    }
}

# Create output directory if it doesn't exist
CONFIG['paths']['output_dir'].mkdir(parents=True, exist_ok=True)

# Create full output prefix path
CONFIG['paths']['full_output_prefix'] = CONFIG['paths']['output_dir'] / CONFIG['paths']['output_prefix']

# %%
# Helper functions for path handling
def get_image_path(filename):
    return str(CONFIG['paths']['image_dir'] / filename)

def get_output_path(filename):
    return str(CONFIG['paths']['output_dir'] / filename)

# Example usage in notebook:
# Single image path
SINGLE_IMAGE_PATH = get_image_path('test2.jpg')

# Directory path
IMAGE_DIR_PATH = str(CONFIG['paths']['image_dir'])

# Output path
OUTPUT_PREFIX = str(CONFIG['paths']['full_output_prefix'])

In [3]:
# Image Processing Class
class ImageProcessor:
    def __init__(self):
        self.resize_width = CONFIG['image']['resize_width']
        self.allowed_extensions = CONFIG['image']['allowed_extensions']
        self.processed_images = {}
    
    def process_single(self, image_path):
        return self.process_image(image_path)
    
    def process_multiple(self, directory_path):
        path = Path(directory_path)
        results = {}
        for img_path in path.glob('*'):
            if img_path.suffix.lower() in self.allowed_extensions:
                results[str(img_path)] = self.process_image(img_path)
        return results
    
    def process_combined(self, directory_path):
        path = Path(directory_path)
        combined_pixel_data = []
        image_ranges = {}
        start_idx = 0
        
        for img_path in path.glob('*'):
            if img_path.suffix.lower() in self.allowed_extensions:
                img_data = self.process_image(img_path)
                pixel_data = img_data['pixel_data']
                combined_pixel_data.append(pixel_data)
                
                end_idx = start_idx + len(pixel_data)
                image_ranges[str(img_path)] = (start_idx, end_idx)
                start_idx = end_idx
        
        return {
            'pixel_data': np.vstack(combined_pixel_data),
            'image_ranges': image_ranges
        }
    
    def process_image(self, image_path):
        img = Image.open(image_path)
        
        # Determine target dimensions based on orientation
        if img.width >= img.height:  # Landscape or square
            # Resize width to target width
            width_percent = (self.resize_width / float(img.size[0]))
            new_height = int((float(img.size[1]) * float(width_percent)))
            target_size = (self.resize_width, new_height)
        else:  # Portrait
            # Resize height to target width (to maintain same scale)
            height_percent = (self.resize_width / float(img.size[1]))
            new_width = int((float(img.size[0]) * float(height_percent)))
            target_size = (new_width, self.resize_width)
        
        # Resize image
        img = img.resize(target_size, Image.Resampling.LANCZOS)
        
        # Create a square canvas with consistent dimensions
        canvas_size = (self.resize_width, self.resize_width)
        canvas = Image.new('RGB', canvas_size, (0, 0, 0))  # Black background
        
        # Calculate position to paste (center the image)
        paste_x = (canvas_size[0] - img.size[0]) // 2
        paste_y = (canvas_size[1] - img.size[1]) // 2
        
        # Paste resized image onto canvas
        canvas.paste(img, (paste_x, paste_y))
        
        # Convert to numpy array and continue processing
        img_array = np.array(canvas)
        
        result = {
            'file_path': str(image_path),
            'file_name': Path(image_path).name,
            'original_size': img.size,
            'original_orientation': 'landscape' if img.width >= img.height else 'portrait',
            'resized_shape': img_array.shape,
            'canvas_shape': canvas_size,
            'array': img_array,
            'pixel_data': self._extract_pixel_data(img_array)
        }
        
        self.processed_images[str(image_path)] = result
        return result
    
    def _extract_pixel_data(self, img_array):
        height, width = img_array.shape[:2]
        y_coords, x_coords = np.meshgrid(np.arange(height), np.arange(width), indexing='ij')
        
        x_normalized = x_coords / width
        y_normalized = y_coords / height
        
        pixels_rgb = img_array.reshape(-1, 3)
        pixel_data = np.column_stack([
            pixels_rgb,
            x_normalized.reshape(-1, 1),
            y_normalized.reshape(-1, 1)
        ])
        
        return pixel_data


In [4]:

# UMAP Processing Class
class UMAPProcessor:
    def __init__(self):
        self.params = CONFIG['umap']
        self.reducers = {
            '2d': umap.UMAP(**self.params['2d']),
            '3d': umap.UMAP(**self.params['3d'])
        }
    
    def process_single(self, pixel_data, filename):
        return self.process(pixel_data, filename)
    
    def process_multiple(self, processed_images):
        results = {}
        for path, data in processed_images.items():
            results[path] = self.process(data['pixel_data'], data['file_name'])
        return results
    
    def process_combined(self, combined_data):
        pixel_data = combined_data['pixel_data']
        results = self.process(pixel_data, "Combined UMAP")
        results['image_ranges'] = combined_data['image_ranges']
        return results
    
    def process(self, pixel_data, filename):
        results = {}
        for dim in ['2d', '3d']:
            embedding = self.reducers[dim].fit_transform(pixel_data)
            if CONFIG['visualization']['plot_umap']:
                self._plot_umap(embedding, pixel_data, filename, dim)
            results[dim] = embedding
        return results
    
    def _plot_umap(self, embedding, pixel_data, filename, dim):
        if dim == '2d':
            plt.figure(figsize=(10, 10))
            plt.scatter(embedding[:, 0], embedding[:, 1],
                       c=pixel_data[:, :3]/255, s=1, alpha=0.5)
            plt.title(f'2D UMAP - {filename}')
            plt.show()
        else:
            fig = plt.figure(figsize=(10, 10))
            ax = fig.add_subplot(111, projection='3d')
            ax.scatter(embedding[:, 0], embedding[:, 1], embedding[:, 2],
                      c=pixel_data[:, :3]/255, s=1, alpha=0.5)
            ax.set_title(f'3D UMAP - {filename}')
            plt.show()

In [5]:
# LED Matrix Generator Class
class LEDMatrixGenerator:
    def __init__(self):
        self.matrix_size = CONFIG['led']['matrix_size']
        self.brightness = CONFIG['led']['brightness']
    
    def generate_header(self, umap_data, pixel_colors, mapping='2d'):
        header = self._get_header_template()
        led_data = self._calculate_led_colors(umap_data[mapping], pixel_colors)
        header += self._format_led_data(led_data)
        header += "};\n\n#endif"
        return header
    
    def _get_header_template(self):
        return f"""// UMAP LED mapping data
#ifndef UMAP_DATA_H
#define UMAP_DATA_H

struct LEDPoint {{
  uint8_t x;
  uint8_t y;
  uint8_t r;
  uint8_t g;
  uint8_t b;
}};

const LEDPoint umapData[{self.matrix_size * self.matrix_size}] = {{
"""
    
    def _calculate_led_colors(self, embedding, pixel_colors):
        embedding_norm = (embedding - embedding.min(axis=0)) / (embedding.max(axis=0) - embedding.min(axis=0))
        led_data = []
        
        for i in range(self.matrix_size):
            for j in range(self.matrix_size):
                umap_x = -25 + (i/7.0) * 50
                umap_y = -25 + (j/7.0) * 50
                
                distances = np.sqrt(
                    (embedding[:, 0] - umap_x)**2 + 
                    (embedding[:, 1] - umap_y)**2
                )
                
                nearest_indices = np.argsort(distances)[:5]
                weights = 1 / (distances[nearest_indices] + 1e-6)
                weights = weights / weights.sum()
                
                colors = np.sum(pixel_colors[nearest_indices] * weights[:, np.newaxis], axis=0)
                colors = colors * (self.brightness / 255)  # Apply brightness
                
                led_data.append({
                    'x': i,
                    'y': j,
                    'r': int(min(255, max(0, colors[0]))),
                    'g': int(min(255, max(0, colors[1]))),
                    'b': int(min(255, max(0, colors[2])))
                })
        
        return led_data
    
    def _format_led_data(self, led_data):
        return ''.join(
            f"  {{{led['x']}, {led['y']}, {led['r']}, {led['g']}, {led['b']}}}"
            f"{','if idx < len(led_data)-1 else ''}\n"
            for idx, led in enumerate(led_data)
        )
    
    def preview_matrix(self, header_content):
        pattern = r'\{(\d+),\s*(\d+),\s*(\d+),\s*(\d+),\s*(\d+)\}'
        matches = re.findall(pattern, header_content)
        
        matrix = np.zeros((self.matrix_size, self.matrix_size, 3), dtype=np.uint8)
        for match in matches:
            x, y, r, g, b = map(int, match)
            matrix[x, y] = [r, g, b]
        
        plt.figure(figsize=(8, 8))
        plt.imshow(matrix)
        plt.title('LED Matrix Preview')
        plt.axis('off')
        plt.show()
    
    def save_header(self, filename, header_content):
        output_path = Path(filename)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        with open(output_path, 'w') as f:
            f.write(header_content)
        print(f"LED matrix data saved to: {output_path}")

In [6]:
# LED Animation Class
class LEDAnimator:
    def __init__(self):
        self.matrix_size = CONFIG['led']['matrix_size']
    
    def generate_pulse_animation(self, umap_data, pixel_colors, frames=30, mapping='2d'):
        """
        Generate pulsing animation for LED matrix
        
        Args:
            umap_data: UMAP embedding data
            pixel_colors: Original pixel colors
            frames: Number of animation frames
            mapping: '2d' or '3d' UMAP mapping
            
        Returns:
            Header content for animation
        """
        # Get embedding data and normalize it
        embedding = umap_data[mapping]
        embedding_norm = (embedding - embedding.min(axis=0)) / (embedding.max(axis=0) - embedding.min(axis=0))
        
        # Calculate LED colors for the base frame using similar logic to LEDMatrixGenerator
        base_frame = np.zeros((self.matrix_size, self.matrix_size, 3), dtype=np.uint8)
        
        # First create the base LED mapping similarly to how LEDMatrixGenerator does it
        for i in range(self.matrix_size):
            for j in range(self.matrix_size):
                # Use the same scaling as in LEDMatrixGenerator to be consistent
                umap_x = -25 + (i/7.0) * 50
                umap_y = -25 + (j/7.0) * 50
                
                # Find nearest points in embedding space
                distances = np.sqrt(
                    (embedding[:, 0] - umap_x)**2 + 
                    (embedding[:, 1] - umap_y)**2
                )
                
                nearest_indices = np.argsort(distances)[:5]
                weights = 1 / (distances[nearest_indices] + 1e-6)
                weights = weights / weights.sum()
                
                # Calculate weighted color
                colors = np.sum(pixel_colors[nearest_indices] * weights[:, np.newaxis], axis=0)
                
                # Set the color in the base frame
                base_frame[j][i] = [
                    int(min(255, max(0, colors[0]))),
                    int(min(255, max(0, colors[1]))),
                    int(min(255, max(0, colors[2])))
                ]
        
        # Now generate all frames with pulsing effect
        all_frames = []
        
        # Scale factors for pulsing
        max_intensity = 1.2
        min_intensity = 0.8
        
        # Generate frames with pulsing effect
        for frame_idx in range(frames):
            # Calculate pulse factor
            pulse_phase = frame_idx / frames * 2 * np.pi
            pulse_factor = min_intensity + (max_intensity - min_intensity) * (np.sin(pulse_phase) + 1) / 2
            
            # Create new frame by applying pulse factor to base frame
            frame = np.zeros_like(base_frame)
            for y in range(self.matrix_size):
                for x in range(self.matrix_size):
                    # Apply pulse factor and ensure values are in valid range
                    frame[y][x] = [
                        int(min(255, max(0, base_frame[y][x][0] * pulse_factor))),
                        int(min(255, max(0, base_frame[y][x][1] * pulse_factor))),
                        int(min(255, max(0, base_frame[y][x][2] * pulse_factor)))
                    ]
            
            all_frames.append(frame)
        
        # Generate header content
        header_content = self._get_animation_template(len(all_frames))
        header_content += self._format_animation_frames(all_frames)
        header_content += self._get_animation_footer()
        
        return header_content
    
    def generate_transition_animation(self, headers, frames_per_transition=30):
        """
        Generate animation transitioning between multiple states
        
        Args:
            headers: List of header contents for each state
            frames_per_transition: Number of frames between states
        """
        # Extract frames from each header
        all_state_frames = []
        for header in headers:
            state_frames = self._extract_frames(header)
            if state_frames:
                all_state_frames.append(state_frames[0])  # Get first frame from each state
        
        if not all_state_frames:
            print("Warning: No valid frames extracted from headers")
            return self._get_animation_template(0) + self._get_animation_footer()
        
        # Generate transitions
        all_frames = []
        for i in range(len(all_state_frames)):
            current_frame = all_state_frames[i]
            next_frame = all_state_frames[(i + 1) % len(all_state_frames)]
            
            # Add current frame
            all_frames.append(current_frame)
            
            # Generate transition frames
            for f in range(frames_per_transition):
                alpha = f / frames_per_transition
                transition_frame = self._interpolate_frames(current_frame, next_frame, alpha)
                all_frames.append(transition_frame)
        
        # Create header content
        header_content = self._get_animation_template(len(all_frames))
        header_content += self._format_animation_frames(all_frames)
        header_content += self._get_animation_footer()
        
        return header_content
    
    def _interpolate_frames(self, frame1, frame2, alpha):
        """Interpolate between two frames"""
        return np.uint8(frame1 * (1 - alpha) + frame2 * alpha)
    
    def _extract_frames(self, header_content):
        """Extract frame data from animation header content"""
        frames = []
        pattern = re.compile(r'\{\s*\{\s*((?:\{\s*\d+\s*,\s*\d+\s*,\s*\d+\s*\}\s*,?\s*)+)\}\s*\}')

        for match in pattern.finditer(header_content):
            frame_data = match.group(1)
            frame = np.zeros((self.matrix_size, self.matrix_size, 3), dtype=np.uint8)

            row_pattern = re.compile(r'\{\s*(\d+)\s*,\s*(\d+)\s*,\s*(\d+)\s*\}')
            for row, row_data in enumerate(frame_data.split('},{')):
                for col, (r, g, b) in enumerate(row_pattern.findall(row_data)):
                    frame[row, col] = [int(r), int(g), int(b)]

            frames.append(frame)

        return frames
    
    def _get_animation_template(self, num_frames):
        """Get animation header template"""
        return f"""// UMAP Animation Data
#ifndef UMAP_ANIMATION_H
#define UMAP_ANIMATION_H

#define NUM_FRAMES {num_frames}
#define MATRIX_SIZE {self.matrix_size}

struct LEDFrame {{
  uint8_t matrix[MATRIX_SIZE][MATRIX_SIZE][3];
}};

const LEDFrame animation[NUM_FRAMES] = {{
"""
    
    def _format_animation_frames(self, frames):
        """Format frames for animation header"""
        output = ""
        for frame_idx, frame in enumerate(frames):
            output += f"  {{\n    {{\n"
            for y in range(self.matrix_size):
                output += "      {"
                for x in range(self.matrix_size):
                    r = min(255, max(0, int(frame[y][x][0])))
                    g = min(255, max(0, int(frame[y][x][1])))
                    b = min(255, max(0, int(frame[y][x][2])))
                    output += f"{{{r},{g},{b}}}"
                    if x < self.matrix_size - 1:
                        output += ","
                output += "}"
                if y < self.matrix_size - 1:
                    output += ",\n      "
                else:
                    output += "\n"
            output += "    }\n  }"
            if frame_idx < len(frames) - 1:
                output += ",\n"
            else:
                output += "\n"
        return output
    
    def _get_animation_footer(self):
        """Get animation footer template"""
        return "};\n\n#endif"
    
    def preview_animation(self, header_content):
        """Preview animation from header content"""
        frames = self._extract_frames(header_content)
        if not frames:
            print("No frames found to preview")
            return
            
        print(f"Previewing animation with {len(frames)} frames")
        
        if CONFIG['visualization']['preview_animation'] and len(frames) > 0:
            # Display first frame as static image
            plt.figure(figsize=(8, 8))
            plt.imshow(frames[0])
            plt.title(f'Animation Preview (Frame 1 of {len(frames)})')
            plt.axis('off')
            plt.show()
    
    def save_animation_header(self, filename, header_content):
        """Save animation header to file"""
        output_path = Path(filename)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        with open(output_path, 'w') as f:
            f.write(header_content)
        print(f"Animation header saved to {output_path}")

In [7]:
# Main processing function
def process_images(input_path, mode=ProcessingMode.SINGLE, custom_output_prefix=None):
    """
    Process images according to specified mode
    
    Args:
        input_path: Path to image or directory
        mode: ProcessingMode enum value
        custom_output_prefix: Optional custom prefix to override CONFIG
        
    Returns:
        Processed results
    """
    # Use custom prefix if provided, otherwise use CONFIG
    output_prefix = custom_output_prefix if custom_output_prefix else CONFIG['paths']['full_output_prefix']
    
    image_proc = ImageProcessor()
    umap_proc = UMAPProcessor()
    led_gen = LEDMatrixGenerator()
    
    if mode == ProcessingMode.SINGLE:
        image_data = image_proc.process_single(input_path)
        umap_data = umap_proc.process_single(image_data['pixel_data'], image_data['file_name'])
        header = led_gen.generate_header(umap_data, image_data['pixel_data'][:, :3])
        if CONFIG['visualization']['preview_matrix']:
            led_gen.preview_matrix(header)
        
        # Use CONFIG for consistent filename
        output_filename = f"{output_prefix}{CONFIG['output']['matrix_suffix']}.h"
        led_gen.save_header(output_filename, header)
        print(f"LED matrix data saved to: {output_filename}")
        
        return {'image_data': image_data, 'umap_data': umap_data}
    
    elif mode == ProcessingMode.MULTIPLE:
        results = {}
        image_data = image_proc.process_multiple(input_path)
        umap_data = umap_proc.process_multiple(image_data)
        
        for idx, (path, data) in enumerate(image_data.items()):
            header = led_gen.generate_header(umap_data[path], data['pixel_data'][:, :3])
            if CONFIG['visualization']['preview_matrix']:
                led_gen.preview_matrix(header)
                
            # Use CONFIG for consistent filename
            output_filename = f"{output_prefix}{CONFIG['output']['multiple_suffix']}_{idx}.h"
            led_gen.save_header(output_filename, header)
            print(f"LED matrix data saved to: {output_filename}")
            
            results[path] = {'image_data': data, 'umap_data': umap_data[path]}
        
        return results
    
    elif mode == ProcessingMode.COMBINED:
        combined_data = image_proc.process_combined(input_path)
        umap_data = umap_proc.process_combined(combined_data)
        header = led_gen.generate_header(umap_data, combined_data['pixel_data'][:, :3])
        if CONFIG['visualization']['preview_matrix']:
            led_gen.preview_matrix(header)
            
        # Use CONFIG for consistent filename
        output_filename = f"{output_prefix}{CONFIG['output']['combined_suffix']}.h"
        led_gen.save_header(output_filename, header)
        print(f"LED matrix data saved to: {output_filename}")
        
        return {'combined_data': combined_data, 'umap_data': umap_data}
    
    else:
        raise ValueError(f"Unknown processing mode: {mode}")

In [18]:
# Animation generation function
def generate_animation(results, mode):
    frames = []
    for key, value in results.items():
        # Debug: Print the key and value being processed
        print(f"Processing {key}: {value}")
        
        # Extract UMAP data
        umap_data = value.get('umap_data', {}).get('2d', None)
        if umap_data is not None:
            # Generate frame from UMAP data
            frame = create_frame_from_umap(umap_data)
            frames.append(frame)
        else:
            print(f"No UMAP data found for {key}")

    if not frames:
        print("Warning: No valid frames extracted from headers")
    
    # Generate animation header
    animation_header = create_animation_header(frames)
    return animation_header

def create_frame_from_umap(umap_data):
    # Example function to create a frame from UMAP data
    frame = {
        'matrix': [[[0, 0, 0] for _ in range(8)] for _ in range(8)]
    }
    # Populate frame with UMAP data (this is just a placeholder logic)
    for i in range(min(len(umap_data), 8)):
        for j in range(min(len(umap_data[i]), 8)):
            frame['matrix'][i][j] = [int(umap_data[i][j] * 255) % 256] * 3
    return frame

def create_animation_header(frames):
    num_frames = len(frames)
    header = f"""
// UMAP Animation Data
#ifndef UMAP_ANIMATION_H
#define UMAP_ANIMATION_H

#define NUM_FRAMES {num_frames}
#define MATRIX_SIZE 8

struct LEDFrame {{
  uint8_t matrix[MATRIX_SIZE][MATRIX_SIZE][3];
}};

const LEDFrame animation[NUM_FRAMES] = {{
"""
    for frame in frames:
        header += "{\n"
        for row in frame['matrix']:
            header += "  {"
            header += ", ".join(f"{{{r[0]}, {r[1]}, {r[2]}}}" for r in row)
            header += "},\n"
        header += "},\n"
    header += """
};

#endif
"""
    return header

def generate_transition_animation(self, headers, frames_per_transition=30):
    """
    Generate animation transitioning between multiple states
    
    Args:
        headers: List of header contents for each state
        frames_per_transition: Number of frames between states
    """
    # Extract LED point data from each header
    all_state_data = []
    for header in headers:
        led_points = self._extract_led_points(header)
        if led_points:
            all_state_data.append(led_points)
    
    if not all_state_data:
        print("Warning: No valid LED points extracted from headers")
        return self._get_animation_template(0) + self._get_animation_footer()
    
    # Generate transitions
    all_frames = []
    for i in range(len(all_state_data)):
        current_points = all_state_data[i]
        next_points = all_state_data[(i + 1) % len(all_state_data)]
        
        # Add current frame
        current_frame = self._led_points_to_frame(current_points)
        all_frames.append(current_frame)
        
        # Generate transition frames
        for f in range(frames_per_transition):
            alpha = f / frames_per_transition
            transition_points = self._interpolate_led_points(current_points, next_points, alpha)
            transition_frame = self._led_points_to_frame(transition_points)
            all_frames.append(transition_frame)
    
    # Create header content
    header_content = self._get_animation_template(len(all_frames))
    header_content += self._format_animation_frames(all_frames)
    header_content += self._get_animation_footer()
    
    return header_content

def _extract_led_points(self, header_content):
    """Extract LED point data from header content"""
    pattern = re.compile(r'\{\s*(\d+)\s*,\s*(\d+)\s*,\s*(\d+)\s*,\s*(\d+)\s*,\s*(\d+)\s*\}')
    matches = pattern.findall(header_content)
    
    led_points = []
    for match in matches:
        x, y, r, g, b = map(int, match)
        led_points.append({'x': x, 'y': y, 'r': r, 'g': g, 'b': b})
    
    return led_points

def _led_points_to_frame(self, led_points):
    """Convert LED points to a frame"""
    frame = np.zeros((self.matrix_size, self.matrix_size, 3), dtype=np.uint8)
    for point in led_points:
        x, y, r, g, b = point['x'], point['y'], point['r'], point['g'], point['b']
        frame[y, x] = [r, g, b]
    return frame

def _interpolate_led_points(self, points1, points2, alpha):
    """Interpolate between two sets of LED points"""
    interpolated_points = []
    for p1, p2 in zip(points1, points2):
        r = int(p1['r'] * (1 - alpha) + p2['r'] * alpha)
        g = int(p1['g'] * (1 - alpha) + p2['g'] * alpha)
        b = int(p1['b'] * (1 - alpha) + p2['b'] * alpha)
        interpolated_points.append({'x': p1['x'], 'y': p1['y'], 'r': r, 'g': g, 'b': b})
    return interpolated_points

In [19]:
# Single image processing
"""
single_results = process_images(
    SINGLE_IMAGE_PATH,
    mode=ProcessingMode.SINGLE
)
animation_header = generate_animation(single_results, mode=ProcessingMode.SINGLE)
"""

# Multiple image processing
multiple_results = process_images(
    IMAGE_DIR_PATH,
    mode=ProcessingMode.MULTIPLE
)

# Debug: Print the results of process_images
print("Multiple Results:", multiple_results)

animation_header = generate_animation(multiple_results, mode=ProcessingMode.MULTIPLE)

# Debug: Print the generated animation header
print("Animation Header:", animation_header)

# Combined image processing
"""
combined_results = process_images(
    IMAGE_DIR_PATH,
    mode=ProcessingMode.COMBINED
)
animation_header = generate_animation(combined_results, mode=ProcessingMode.COMBINED)
"""

/Users/matthewlaujh/Desktop/desktop/itpnyu/itpnyu-thesis/itpnyu-thesis/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/matthewlaujh/Desktop/desktop/itpnyu/itpnyu-thesis/itpnyu-thesis/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/Users/matthewlaujh/Desktop/desktop/itpnyu/itpnyu-thesis/itpnyu-thesis/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/matthewlaujh/Desktop/desktop/itpnyu/itpnyu-thesis/itpnyu-thesis/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/Users/matthew

LED matrix data saved to: /Users/matthewlaujh/Desktop/desktop/itpnyu/itpnyu-thesis/itpnyu-thesis/prod/output/fenway/umap_data_multiple_0.h
LED matrix data saved to: /Users/matthewlaujh/Desktop/desktop/itpnyu/itpnyu-thesis/itpnyu-thesis/prod/output/fenway/umap_data_multiple_0.h
LED matrix data saved to: /Users/matthewlaujh/Desktop/desktop/itpnyu/itpnyu-thesis/itpnyu-thesis/prod/output/fenway/umap_data_multiple_1.h
LED matrix data saved to: /Users/matthewlaujh/Desktop/desktop/itpnyu/itpnyu-thesis/itpnyu-thesis/prod/output/fenway/umap_data_multiple_1.h
LED matrix data saved to: /Users/matthewlaujh/Desktop/desktop/itpnyu/itpnyu-thesis/itpnyu-thesis/prod/output/fenway/umap_data_multiple_2.h
LED matrix data saved to: /Users/matthewlaujh/Desktop/desktop/itpnyu/itpnyu-thesis/itpnyu-thesis/prod/output/fenway/umap_data_multiple_2.h
Multiple Results: {'/Users/matthewlaujh/Desktop/desktop/itpnyu/itpnyu-thesis/itpnyu-thesis/prod/images/fenway/R0011041.JPG': {'image_data': {'file_path': '/Users/matt

'\ncombined_results = process_images(\n    IMAGE_DIR_PATH,\n    mode=ProcessingMode.COMBINED\n)\nanimation_header = generate_animation(combined_results, mode=ProcessingMode.COMBINED)\n'